# Custom mode

This notebook builds a small custom-mode panel. It keeps a selected headlight color applied for a fixed duration and announces the selected mode name through the robot speaker.

Run the cells from top to bottom while the robot is connected to the same DDS network. Keep the announcements short because they are spoken out loud.


Choose the DDS interface. This notebook uses only the Unitree Python SDK plus local standard-library helpers.


In [11]:
import os

IFACE = os.environ.get("G1_IFACE", "eth0")
DOMAIN_ID = int(os.environ.get("G1_DOMAIN_ID", "0"))
print(f"Configured for iface={IFACE!r}, domain_id={DOMAIN_ID}.")


Configured for iface='eth0', domain_id=0.


Import the direct Unitree audio client and notebook UI helpers.


In [12]:
import audioop
import re
import shutil
import subprocess
import tempfile
import threading
import time
import wave
from pathlib import Path

import ipywidgets as widgets
from IPython.display import display
from unitree_sdk2py.core.channel import ChannelFactoryInitialize
from unitree_sdk2py.g1.audio.g1_audio_client import AudioClient


Create a small direct audio/headlight client. It initializes DDS, calls `AudioClient.LedControl()` for the headlight, and uses Piper plus `AudioClient.PlayStream()` for speech.


In [13]:
NAMED_COLORS = {
    "white": (255, 255, 255),
    "red": (255, 0, 0),
    "green": (0, 255, 0),
    "blue": (0, 0, 255),
    "yellow": (255, 255, 0),
    "cyan": (0, 255, 255),
    "magenta": (255, 0, 255),
    "orange": (255, 165, 0),
    "purple": (128, 0, 128),
    "pink": (255, 105, 180),
}

PIPER_VOICE_DIR = Path.home() / ".local" / "share" / "piper" / "voices"
DEFAULT_PIPER_VOICES = {
    "en": "en_US-lessac-medium",
    "en_us": "en_US-lessac-medium",
    "english": "en_US-lessac-medium",
    "de": "de_DE-thorsten-medium",
    "de_de": "de_DE-thorsten-medium",
    "german": "de_DE-thorsten-medium",
    "fr": "fr_FR-siwis-medium",
    "fr_fr": "fr_FR-siwis-medium",
    "french": "fr_FR-siwis-medium",
    "es": "es_ES-davefx-medium",
    "es_es": "es_ES-davefx-medium",
    "spanish": "es_ES-davefx-medium",
}


def parse_color(value):
    if isinstance(value, tuple) and len(value) == 3:
        return tuple(int(max(0, min(255, v))) for v in value)

    lowered = str(value).strip().lower()
    if lowered in NAMED_COLORS:
        return NAMED_COLORS[lowered]
    if re.fullmatch(r"#?[0-9a-fA-F]{6}", lowered):
        hexval = lowered.lstrip("#")
        return (int(hexval[0:2], 16), int(hexval[2:4], 16), int(hexval[4:6], 16))
    if re.fullmatch(r"\d{1,3},\d{1,3},\d{1,3}", lowered):
        parts = [int(p) for p in lowered.split(",")]
        if all(0 <= p <= 255 for p in parts):
            return (parts[0], parts[1], parts[2])
    raise ValueError("color must be a name, #RRGGBB, or R,G,B")


def scale_color(rgb, intensity):
    level = max(0, min(100, int(intensity)))
    if level >= 100:
        return rgb
    scale = level / 100.0
    return (int(rgb[0] * scale), int(rgb[1] * scale), int(rgb[2] * scale))


def piper_voice_model_path(voice_name):
    return PIPER_VOICE_DIR / voice_name / f"{voice_name}.onnx"


def resolve_piper_model(model=None, language=None):
    value = model
    if not value and language:
        voice_name = DEFAULT_PIPER_VOICES.get(str(language).strip().lower().replace("-", "_"))
        if not voice_name:
            raise ValueError("unsupported Piper language; supported: en, de, fr, es")
        value = piper_voice_model_path(voice_name)
    if not value:
        value = os.environ.get("G1_PIPER_MODEL") or os.environ.get("PIPER_MODEL")
    if not value:
        value = piper_voice_model_path("en_US-lessac-medium")

    model_path = Path(value).expanduser()
    if not model_path.exists():
        raise FileNotFoundError(f"Piper voice model does not exist: {model_path}")
    return model_path


def convert_wav_for_robot(src_path, dst_path):
    with wave.open(str(src_path), "rb") as wf:
        channels = wf.getnchannels()
        sample_width = wf.getsampwidth()
        frame_rate = wf.getframerate()
        pcm = wf.readframes(wf.getnframes())

    if channels == 2:
        pcm = audioop.tomono(pcm, sample_width, 0.5, 0.5)
        channels = 1
    elif channels != 1:
        raise ValueError(f"WAV must be mono or stereo PCM, got {channels} channels")

    if sample_width != 2:
        pcm = audioop.lin2lin(pcm, sample_width, 2)
        sample_width = 2

    if frame_rate != 16000:
        pcm, _state = audioop.ratecv(pcm, sample_width, channels, frame_rate, 16000, None)

    with wave.open(str(dst_path), "wb") as wf:
        wf.setnchannels(1)
        wf.setsampwidth(2)
        wf.setframerate(16000)
        wf.writeframes(pcm)
    return dst_path


class DirectRobotAudio:
    def __init__(self, iface, domain_id):
        ChannelFactoryInitialize(int(domain_id), str(iface))
        self.client = AudioClient()
        self.client.SetTimeout(5.0)
        self.client.Init()

    def headlight(self, color="white", intensity=100):
        rgb = scale_color(parse_color(color), intensity)
        return int(self.client.LedControl(*rgb))

    def say(self, text, volume=None, language=None, voice_model=None, speaker=None):
        piper_bin = os.environ.get("G1_PIPER_BIN") or os.environ.get("PIPER_BIN") or "piper"
        piper_path = shutil.which(piper_bin)
        if piper_path is None:
            local_piper = Path.home() / ".local" / "bin" / "piper"
            if piper_bin == "piper" and local_piper.exists():
                piper_path = str(local_piper)
            else:
                raise RuntimeError("piper is required for speech; set G1_PIPER_BIN or PIPER_BIN")

        model_path = resolve_piper_model(voice_model, language=language)
        if volume is not None:
            code = int(self.client.SetVolume(int(volume)))
            if code != 0:
                return code

        with tempfile.TemporaryDirectory(prefix="g1_say_") as td:
            wav_path = Path(td) / "speech.wav"
            robot_wav_path = Path(td) / "speech_robot.wav"
            command = [piper_path, "--model", str(model_path), "--output-file", str(wav_path)]
            if speaker is not None:
                command.extend(["--speaker", str(int(speaker))])
            subprocess.run(command, input=str(text), text=True, check=True)
            with wave.open(str(convert_wav_for_robot(wav_path, robot_wav_path)), "rb") as wf:
                pcm = wf.readframes(wf.getnframes())
            code, _data = self.client.PlayStream("custom_mode", "custom-mode-1", pcm)
            return int(code)


robot_audio = DirectRobotAudio(IFACE, DOMAIN_ID)
print("Direct Unitree audio client ready.")


Direct Unitree audio client ready.


The controller stores the desired mode name and color, resends the headlight command every 0.2 seconds for the configured duration, announces the mode name when started, and announces the end when the duration expires or Stop is clicked.


In [14]:
HEADLIGHT_RESEND_INTERVAL_S = 0.2


class CustomModeController:
    def __init__(self, audio, duration_s=10.0):
        self.audio = audio
        self.duration_s = float(duration_s)
        self.mode_name = "academy custom mode"
        self.color = "#1e90ff"
        self.intensity = 80
        self.enabled = False
        self._lock = threading.RLock()
        self._stop = threading.Event()
        self._thread = None
        self._deadline = None

    def start(self):
        with self._lock:
            self.enabled = True
            self._stop.clear()
            self._deadline = time.monotonic() + self.duration_s
            light_code = self._apply_locked()
            speech_code = self._announce_start_locked()
            if self._thread is None or not self._thread.is_alive():
                self._thread = threading.Thread(target=self._loop, daemon=True)
                self._thread.start()
        return (
            f"Custom mode is running for {self.duration_s:.1f}s. "
            f"Headlight rc={light_code}; speech rc={speech_code}."
        )

    def stop(self):
        with self._lock:
            if not self.enabled:
                return "Custom mode is already stopped."
            self.enabled = False
            self._stop.set()
            speech_code = self._announce_end_locked()
        return f"Custom mode stopped early; end announcement rc={speech_code}."

    def update(self, mode_name=None, color=None, intensity=None, duration_s=None):
        with self._lock:
            if mode_name is not None:
                self.mode_name = str(mode_name).strip() or self.mode_name
            if color is not None:
                self.color = str(color)
            if intensity is not None:
                self.intensity = max(0, min(100, int(intensity)))
            if duration_s is not None:
                self.duration_s = max(0.2, float(duration_s))
                if self.enabled:
                    self._deadline = time.monotonic() + self.duration_s
            if self.enabled:
                self._apply_locked()
        return self.summary()

    def summary(self):
        return (
            f"mode={self.mode_name!r} color={self.color} intensity={self.intensity} "
            f"duration={self.duration_s:.1f}s resend={HEADLIGHT_RESEND_INTERVAL_S:.1f}s enabled={self.enabled}"
        )

    def _announce_start_locked(self):
        return self.audio.say(f"Custom mode active: {self.mode_name}")

    def _announce_end_locked(self):
        return self.audio.say(f"Custom mode ended: {self.mode_name}")

    def _apply_locked(self):
        return self.audio.headlight(color=self.color, intensity=self.intensity)

    def _loop(self):
        while True:
            if self._stop.wait(HEADLIGHT_RESEND_INTERVAL_S):
                return

            with self._lock:
                if not self.enabled:
                    return
                if self._deadline is not None and time.monotonic() >= self._deadline:
                    self.enabled = False
                    self._stop.set()
                    try:
                        self._announce_end_locked()
                    finally:
                        return
                try:
                    self._apply_locked()
                except Exception:
                    pass

custom_mode = CustomModeController(robot_audio)
print(custom_mode.summary())


mode='academy custom mode' color=#1e90ff intensity=80 duration=10.0s resend=0.2s enabled=False


Run the panel. Use Start to begin the timed color refresh and announce the current mode; use Stop to end it early.


In [15]:
mode_name = widgets.Text(value="academy custom mode", description="Mode", layout=widgets.Layout(width="420px"))
color = widgets.ColorPicker(value="#1e90ff", description="Color")
intensity = widgets.IntSlider(value=80, min=0, max=100, step=5, description="Intensity")
duration_s = widgets.FloatSlider(value=10.0, min=0.2, max=60.0, step=0.2, description="Duration s")
start_button = widgets.Button(description="Start", button_style="success")
stop_button = widgets.Button(description="Stop", button_style="warning")
status = widgets.HTML(value="")


def sync_settings():
    return custom_mode.update(mode_name.value, color.value, intensity.value, duration_s.value)


def on_start(_):
    try:
        sync_settings()
        status.value = custom_mode.start()
    except Exception as exc:
        status.value = f"Start failed: {exc}"


def on_stop(_):
    try:
        status.value = custom_mode.stop()
    except Exception as exc:
        status.value = f"Stop failed: {exc}"

for widget in (mode_name, color, intensity, duration_s):
    widget.observe(lambda _change: setattr(status, "value", sync_settings()), names="value")
start_button.on_click(on_start)
stop_button.on_click(on_stop)
status.value = custom_mode.summary()
display(widgets.VBox([mode_name, widgets.HBox([color, intensity, duration_s]), widgets.HBox([start_button, stop_button]), status]))
